# Vanilla TASO baseline

Select a GPU runtime. This notebook has not been executed on Colab. It first checks the environment, then attempts an upstream source build and runs one tiny graph. No LLM is involved. Legacy CUDA/cuDNN compatibility may prevent compilation; keep the first failure log for diagnosis.

In [ ]:
import sys, platform, shutil, subprocess
print(platform.platform(), sys.version)
for command in (["nvidia-smi"], ["nvcc", "--version"]):
    result = subprocess.run(command, capture_output=True, text=True) if shutil.which(command[0]) else None
    print(result.stdout + result.stderr if result else f"Missing: {command[0]}")
assert shutil.which("nvidia-smi"), "Select a GPU runtime before continuing."


## Clone this project

The source and baseline script come directly from the project repository.


In [ ]:
from pathlib import Path
work = Path("/content/CS521_Proj")
remote = "https://github.com/venkat0011/CS521_Proj.git"
if not work.exists():
    subprocess.run(["git", "clone", remote, str(work)], check=True)
else:
    print("Using existing checkout; it has not been reset or overwritten.")
repo = work / "taso" / "TASO"
baseline = work / "taso" / "baseline"
print("Project:", work)
print("TASO:", repo)


## Check cuDNN before building

The checked-in Docker environment used cuDNN 7.6. This attempt requires existing cuDNN 7/8 development headers and libraries. It stops for cuDNN 9 rather than claiming compatibility. If it stops, share the output so the environment can be adapted.

In [ ]:
import re
headers = list(Path("/usr/include").glob("**/cudnn_version.h"))
headers += list(Path("/usr/local/cuda/include").glob("cudnn_version.h"))
headers += list(Path("/usr/local/lib").glob("python*/dist-packages/nvidia/cudnn/include/cudnn_version.h"))
headers += list(Path("/usr/local/lib").glob("python*/site-packages/nvidia/cudnn/include/cudnn_version.h"))
print("cuDNN headers:", [str(p) for p in headers])
compatible = []
for header in headers:
    match = re.search(r"#define\s+CUDNN_MAJOR\s+(\d+)", header.read_text())
    if match and int(match[1]) in (7, 8): compatible.append(header)
assert compatible, "Need a compatible cuDNN 7/8 development environment. Save this output; setup is not yet complete."
include = compatible[0].parent
roots = [include.parent / "lib", Path("/usr/lib/x86_64-linux-gnu"), Path("/usr/local/cuda/lib64")]
libs = [p for root in roots for p in root.glob("libcudnn.so*")]
assert libs, "cuDNN headers found but library missing."
cudnn = next((p for p in libs if p.name == "libcudnn.so"), libs[0])
print("Selected:", include, cudnn)


## Build attempt

This installs build dependencies inside the temporary Colab runtime. These pins are a starting point for the legacy checkout, not a verified Colab environment. Stop at the first error.

In [ ]:
def run(command, cwd=None):
    print("Running:", command)
    subprocess.run(command, cwd=cwd, check=True)
run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "build-essential", "libprotobuf-dev", "protobuf-compiler"])
run([sys.executable, "-m", "pip", "install", "cmake==3.26.4", "Cython==0.29.37", "numpy<2", "onnx==1.16.2", "setuptools<81"])
build = repo / "build"
build.mkdir(exist_ok=True)
run(["cmake", "..", "-DUSE_CUDA=/usr/local/cuda", f"-DCMAKE_INCLUDE_PATH={include}", f"-DCUDA_CUDNN_LIBRARY={cudnn}"], cwd=build)
run(["cmake", "--build", ".", "-j2"], cwd=build)
run(["cmake", "--install", "."], cwd=build)
run(["ldconfig"])
run([sys.executable, "setup.py", "install"], cwd=repo / "python")


## Run Input → Conv → ReLU

The child process avoids notebook import-cache issues and captures TASO’s native search output. This establishes whether vanilla TASO runs; internal per-rule profiling comes after this succeeds.

In [ ]:
import os, json
env = dict(os.environ)
env["TASO_HOME"] = str(repo)
env["LD_LIBRARY_PATH"] = ":".join([str(cudnn.parent), "/usr/local/lib", env.get("LD_LIBRARY_PATH", "")])
results = work / "results"
results.mkdir(exist_ok=True)
with (results / "search.log").open("w") as log:
    result = subprocess.run([sys.executable, str(baseline / "run_baseline.py"), "--repo", str(repo), "--out", str(results)], cwd=work, env=env, stdout=log, stderr=subprocess.STDOUT)
print((results / "search.log").read_text())
assert result.returncode == 0, "Baseline failed; inspect search.log."
print(json.loads((results / "baseline.json").read_text()))


In [ ]:
shutil.make_archive("/content/taso_baseline_results", "zip", results)
from google.colab import files
files.download("/content/taso_baseline_results.zip")
